# LangChain Structured Output

> **Key idea:** Structured output asks a language model to return data that follows a defined schema instead of returning only free-form text.

This is useful when the response will be parsed, validated, stored, or passed to another part of an application.

## Why use structured output?

Without a schema, a model may return different wording or formatting for the same request. A schema makes the expected fields and data types explicit.

Benefits include:

- predictable response fields
- automatic parsing into Python objects
- validation of types and required values
- easier integration with APIs, databases, and workflows
- fewer errors caused by manually parsing model text

## The main API

Chat models can be instructed to return a schema by using `with_structured_output()`:

```python
structured_model = model.with_structured_output(ResponseSchema)
result = structured_model.invoke("Extract the person's name and age.")
```

The returned value is usually a validated object or dictionary rather than a plain string.

## Supported schema styles

### 1. Pydantic model

Pydantic is the most useful choice when runtime validation is important.

```python
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str = Field(description="The person's full name")
    age: int = Field(description="The person's age in years")

structured_model = model.with_structured_output(Person)
result = structured_model.invoke("Alice is 30 years old.")

print(result.name)
print(result.age)
```

The result is a `Person` instance. Pydantic validates the field types and raises a validation error when the data cannot match the schema.

### 2. `TypedDict`

`TypedDict` describes the expected dictionary keys and value types. It is lightweight, but it does not provide the same runtime validation as a Pydantic model.

```python
from typing_extensions import TypedDict

class PersonDict(TypedDict):
    name: str
    age: int

structured_model = model.with_structured_output(PersonDict)
result = structured_model.invoke("Alice is 30 years old.")
```

The result is dictionary-like:

```python
print(result["name"])
print(result["age"])
```

### 3. Dataclass

A dataclass is useful when you want a simple Python data container with named fields.

```python
from dataclasses import dataclass

@dataclass
class PersonData:
    name: str
    age: int

structured_model = model.with_structured_output(PersonData)
result = structured_model.invoke("Alice is 30 years old.")
```

### 4. JSON Schema or dictionary schema

A dictionary can describe a JSON Schema when a provider-specific schema is needed:

```python
schema = {
    "title": "Person",
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer"},
    },
    "required": ["name", "age"],
}

structured_model = model.with_structured_output(schema)
```

## How it works

1. Define the expected output schema.
2. Bind the schema to the chat model with `with_structured_output()`.
3. Send a normal prompt with `invoke()` or `batch()`.
4. The model generates structured data, often through provider-supported tool/function calling.
5. LangChain parses the response into the selected Python type.
6. The application uses the validated result.

## Structured output versus tools

Structured output describes the **final data returned by the model**. A tool is a function the model can **request to execute**.

- Use structured output when you need a predictable answer format.
- Use tools when the model needs to perform an action or access external data.
- A tool call may itself use a schema for its arguments.

## Important options

```python
structured_model = model.with_structured_output(
    Person,
    include_raw=True,
)
```

With `include_raw=True`, the result includes both the original model message and the parsed output. This is useful for debugging parsing problems.

## Important revision points

- `with_structured_output()` is the standard LangChain interface.
- Pydantic is preferred when strong runtime validation is required.
- `TypedDict` is convenient for dictionary-shaped results.
- Dataclasses provide a simple object-shaped result.
- Field names, type hints, descriptions, and required fields guide the model.
- The model provider must support structured output or tool/function calling.
- Structured output improves reliability, but the result should still be validated before critical use.


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model
model = init_chat_model(
    model = "groq:openai/gpt-oss-20b"
)

# Response without Structure
print(model.invoke("Tell me about the movie Titanic").content)

**Titanic (1997)** – a romantic‑drama‑disaster epic directed by James Cameron

| Item | Details |
|------|---------|
| **Release** | December 19, 1997 (U.S.) |
| **Runtime** | 195 min (3 h 15 min) |
| **Genre** | Romance, Drama, Historical, Thriller |
| **Budget** | ≈ $200 million |
| **Box‑office** | > $2.2 billion worldwide (the highest‑grossing film of all time until it was surpassed by *Avatar* in 2009, and again reclaimed the title in 2010 after *Avengers: Endgame*). |
| **Director** | James Cameron |
| **Screenplay** | James Cameron, Jon Zack, and James Cameron (story) |
| **Music** | James Horner – Oscar‑winning score (includes the theme “My Heart Will Go On”) |
| **Cinematography** | Russell Miller |
| **Production companies** | Paramount Pictures, 20th Century Fox, Cameron Film Works |
| **Distributor** | Paramount Pictures |

### Plot (≈ 3 sentences)
On the ill‑fated maiden voyage of the RMS Titanic, a young, aristocratic woman, **Rose de Wittman** (Kate Winslet), escapes the

# 1. Pydantic

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie")

model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output.invoke("Tell me about the movie Titanic")


Movie(title='Titanic', year=1997, director='James Cameron', rating=8.1)

In [4]:
# Include raw output

model_with_structured_output = model.with_structured_output(Movie, include_raw=True)
model_with_structured_output.invoke("Tell me about the movie Titanic")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "User wants info about the movie Titanic. We can provide a summary. Maybe also use function? The function requires director, rating, title, year. We could call the function to get a structured response. The function likely returns data. Let's call it.", 'tool_calls': [{'id': 'fc_cc332294-5667-4538-a812-e88eda053f10', 'function': {'arguments': '{"director":"James Cameron","rating":7.8,"title":"Titanic","year":1997}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 155, 'total_tokens': 246, 'completion_time': 0.092057586, 'completion_tokens_details': {'reasoning_tokens': 52}, 'prompt_time': 0.008789774, 'prompt_tokens_details': None, 'queue_time': 0.30579297, 'total_time': 0.10084736}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_565badff47', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provide

#### Nested Structure

In [5]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str = Field(description="The name of the actor")
    age: int = Field(description="The age of the actor")

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie")
    cast: list[Actor] = Field(description="The cast of the movie")

model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output.invoke("Tell me about the movie Titanic")


Movie(title='Titanic', year=1997, director='James Cameron', rating=7.8, cast=[Actor(name='Leonardo DiCaprio', age=39), Actor(name='Kate Winslet', age=39), Actor(name='Billy Zane', age=61), Actor(name='Kathy Bates', age=73), Actor(name='Frances Fisher', age=63)])

# 2. TypeDict

In [13]:
from typing_extensions import TypedDict, Annotated

class ActorDetails(TypedDict):
    name: Annotated[str, "The name of the actor"]
    age: Annotated[int, "The age of the actor"]

class MovieDetails(TypedDict):
    title: Annotated[str, "The title of the movie"]
    year: Annotated[int, "The year the movie was released"]
    director: Annotated[str, "The director of the movie"]
    rating: Annotated[float, "The rating of the movie"]
    cast: list[ActorDetails]

model_with_typedict = model.with_structured_output(MovieDetails)
model_with_typedict.invoke("Tell me about the movie Titanic")

{'cast': [{'age': 26, 'name': 'Leonardo DiCaprio'},
  {'age': 24, 'name': 'Kate Winslet'},
  {'age': 37, 'name': 'Billy Zane'},
  {'age': 45, 'name': 'Kathy Bates'},
  {'age': 48, 'name': 'Frances Fisher'}],
 'director': 'James Cameron',
 'rating': 7.8,
 'title': 'Titanic',
 'year': 1997}

In [15]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

# 3. DataClass

In [16]:
from dataclasses import dataclass

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str

model_with_dataclass = model.with_structured_output(ContactInfo)

model_with_dataclass.invoke("Provide contact information for John Doe, including name, email, and phone number.")
 

{'email': 'john.doe@example.com', 'name': 'John Doe', 'phone': '555-123-4567'}